## Libraries

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import math 
from random import random 
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path 
import librosa 
import json 
from collections import defaultdict
from torchvision.transforms import ToTensor
from torchvision import datasets
from torch.utils.data import Subset
import torchaudio 

## Set up GPU 

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

Device: mps


## Setup

In [3]:
SEED = 42 
torch.manual_seed(SEED)
np.random.seed(SEED)

## Data

### KAGGLE Urban Sounds Dataset

In [4]:
import kagglehub

# Download latest version
download_path = Path(kagglehub.dataset_download("rupakroy/urban-sound-8k"))
root_dir = download_path / "UrbanSound8K" / "UrbanSound8K"
annotations_file = root_dir / "metadata" / "UrbanSound8K.csv"
audio_dir = root_dir / "audio"
print(f"download path: {download_path}\n")
# print(f"data contents: {list(download_path.iterdir())}\n")
# print(f"annotations: {annotations_file}\n")
# print(f"audio dir: {audio_dir}\n")
# print(f"audio dir contents: {list(audio_dir.iterdir())}\n")

download path: /Users/pranavrajan/.cache/kagglehub/datasets/rupakroy/urban-sound-8k/versions/1



In [5]:
df = pd.read_csv(annotations_file)
print(f"shape: {df.shape}")
print(f"column names: {df.columns}")
print(f"column types: {df.dtypes}")
print(f"classes: {df['class'].unique().tolist()}")

shape: (8732, 8)
column names: Index(['slice_file_name', 'fsID', 'start', 'end', 'salience', 'fold',
       'classID', 'class'],
      dtype='object')
column types: slice_file_name     object
fsID                 int64
start              float64
end                float64
salience             int64
fold                 int64
classID              int64
class               object
dtype: object
classes: ['dog_bark', 'children_playing', 'car_horn', 'air_conditioner', 'street_music', 'gun_shot', 'siren', 'engine_idling', 'jackhammer', 'drilling']


In [6]:
df.head()

,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing


### UrbanSound Data Preprocessing

In [7]:
train_folds = [i for i in range(1, 9)]
valid_folds = [9]
test_folds = [10]

train_annotations = df[df["fold"].isin(train_folds)].reset_index(drop=True)
valid_annotations = df[df["fold"].isin(valid_folds)].reset_index(drop=True)
test_annotations  = df[df["fold"].isin(test_folds)].reset_index(drop=True)

print(f"train split samples: {len(train_annotations)}")
print(f"validation split samples: {len(valid_annotations)}")
print(f"test split samples: {len(test_annotations)}")

train split samples: 7079
validation split samples: 816
test split samples: 837


## Dataset Processing

### Constants

In [8]:
SAMPLE_RATE = 22050
FRAME_SIZE = 1024
HOP_LENGTH = 512 
NUM_MELS = 64
NUM_SAMPLES = 22050

### Transformation - Mel-Spectrogram

In [9]:
"""
transformation are callable modules 
"""
# mel_spectrogram = torchaudio.transforms.MelSpectrogram(
#     sample_rate=SAMPLE_RATE, 
#     n_fft=FRAME_SIZE, 
#     hop_length=HOP_LENGTH,
#     n_mels=NUM_MELS
# )

def make_mel():
    return torchaudio.transforms.MelSpectrogram(
        sample_rate=SAMPLE_RATE, 
        n_fft=FRAME_SIZE, 
        hop_length=HOP_LENGTH, 
        n_mels=NUM_MELS
    )


In [10]:
from torchcodec.decoders import AudioDecoder
from torch.utils.data import Dataset

class UrbanSoundDataset(Dataset):
    """
    annotations_file - all the annotations 
    audio_dir - path to all the audio samples in the dataset 
    """
    def __init__(
        self, 
        annotations_file, 
        audio_dir, 
        transformation, 
        target_sample_rate, 
        num_samples,
        device
    ):
        self.device = device
        self.audio_dir = Path(audio_dir)
        self.transformation = transformation.to(self.device)
        self.target_sample_rate = target_sample_rate
        self.num_samples = num_samples
        
        if isinstance(annotations_file, pd.DataFrame):
            self.annotations = annotations_file
        else:
            self.annotations = pd.read_csv(annotations_file)
    

    """
    return number of samples in the dataset
    """
    def __len__(self):
        return len(self.annotations)

    """
    a_list[1] -> a_list.__getitem__(1)
    - not all audio samples are mono-audio samples(1 channel), stereo(2 channels), n-channel(2+ channels)
    """
    def __getitem__(self, index):
        audio_sample_path = self._get_audio_sample_path(index)
        label = self._get_audio_sample_label(index)

        decoder = AudioDecoder(str(audio_sample_path))
        signal = decoder.get_all_samples()
        
        sr = signal.sample_rate 
        samples = signal.data
        samples = samples.to(self.device)
        # print(f"After loading: {samples.shape}, sr={sr}")
        
        # normalize time dimension 
        samples = self._resample_if_necessary(samples, sr)
        # print(f"After resample: {samples.shape}")
        
        # normalize channel dimension 
        samples = self._mix_down_if_necessary(samples)
        # print(f"After mixdown: {samples.shape}")

        # normalize signal length 
        samples = self._cut_if_necessary(samples)
        # print(f"After cutting: {samples.shape}")
        samples = self._right_pad_if_necessary(samples)
        # print(f"After padding: {samples.shape}")

        # apply transformation(output is uniform for deep learning)
        samples = self.transformation(samples)

        return samples, label

    """
    signal -> Tensor -> (1, num_samples)
    - more samples than expected number of samples 
    """
    def _cut_if_necessary(self, samples):
        if samples.shape[1] > self.num_samples:
            samples = samples[:, :self.num_samples]
        return samples 

    """
    [1, 1, 1] -> [1, 1, 1, 0, 0]
    
    """
    def _right_pad_if_necessary(self, samples):
        length_signal = samples.shape[1]
        if length_signal < self.num_samples:
            num_missing_samples = self.num_samples - length_signal
            last_dim_padding = (0, num_missing_samples)
            samples = torch.nn.functional.pad(samples, last_dim_padding)
        return samples 

    """
    make sure all recording samples have the same sample rate for mel spectrograms.
    """
    def _resample_if_necessary(self, samples, sr):
        if sr != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.target_sample_rate).to(self.device)
            samples = resampler(samples)
        return samples

    """
    make sure all recording samples have the same number of channels (mono channel) for deep learning
    """
    def _mix_down_if_necessary(self, samples):
        if samples.shape[0] > 1:
            samples = torch.mean(samples, dim=0, keepdim=True)
        return samples 

    """
    return path to audio file 
    """
    def _get_audio_sample_path(self, index):
        row = self.annotations.iloc[index]
        fold = f"fold{row['fold']}"                  # named col instead of [index, 5]
        return self.audio_dir / fold / row['slice_file_name']

    """
    return the label for a particular audio file 
    """
    def _get_audio_sample_label(self, index):
        return self.annotations.iloc[index]['classID']  # named col instead of [index, 6]

In [11]:
# usd = UrbanSoundDataset(
#     annotations_file, 
#     audio_dir, 
#     mel_spectrogram, 
#     SAMPLE_RATE,
#     NUM_SAMPLES,
#     device
# )
# print(f"There are {len(usd)} samples in the dataset.")
# signal, label = usd[0]
# print(f"Signal shape: {signal.shape} | Label: {label}")

## Neural Network Architecture - Convolutional Neural Network(CNN)

In [12]:
# neural network architecture 
class UrbanSoundCNN(nn.Module):
    def __init__(self, num_classes=10, dropout_prob=0.3):
        super().__init__()
        # build a model
        self.net = nn.Sequential(
            # 1st conv layer
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, ceil_mode=True),
            
            # 2nd conv layer
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, ceil_mode=True),
            
            # 3rd conv layer
            nn.Conv2d(32, 64, kernel_size=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            
            # flatten the output and feed into dense layer
            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            
            # output layer - raw logits - CrossEntropyLoss applies softmax internally
            nn.Linear(64, num_classes),
        )
        
    def forward(self, x):
        return self.net(x)

def train(model, train_dataloader, valid_dataloader, optimizer, loss_fn, device, epochs=50):
    history = defaultdict(list)
    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for x_batch, y_batch in train_dataloader:
            # get a sample of data 
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            # set the gradients to 0 
            optimizer.zero_grad()
            
            # run forward pass
            preds = model(x_batch)
            
            # run cross entropy loss
            loss = loss_fn(preds, y_batch)
            
            # run backward pass
            loss.backward()
            
            # update parameters
            optimizer.step()
 
            # accumulate training loss
            train_loss += loss.item() * x_batch.size(0)
 
            # number of correct predictions in batch
            train_correct += (preds.argmax(1) == y_batch).sum().item()
 
            # number of samples in the batch
            train_total += y_batch.size(0)
 
        # check the model accuracy once per epoch on the validation data
        valid_loss, valid_acc = evaluate(model, valid_dataloader, loss_fn, device)
 
        # update the training statistics
        history["train_loss"].append(train_loss / train_total)
        history["train_acc"].append(train_correct / train_total)
        history["valid_loss"].append(valid_loss)
        history["valid_acc"].append(valid_acc)
 
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss: {train_loss / train_total:.4f} - train_acc: {train_correct / train_total:.4f} | "
            f"valid_loss: {valid_loss:.4f} - valid_acc: {valid_acc:.4f}"
        )
 
    print("---------------------------------------")
    print("Training Finished!\n")

    return history


def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = model(x_batch)
            loss = loss_fn(preds, y_batch)
            total_loss += loss.item() * x_batch.size(0)
            correct += (preds.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)
 
    return total_loss / total, correct / total

## Overfitting Debugging

In [13]:
def plot_history(history):
    fig, axs = plt.subplots(2, figsize=(10, 8))
 
    # create accuracy subplot
    axs[0].plot(history["train_acc"], label="train accuracy")
    axs[0].plot(history["valid_acc"], label="valid accuracy")
    axs[0].set_ylabel("Accuracy")
    axs[0].legend(loc="best")
    axs[0].set_title("Accuracy eval")
 
    # create error subplot
    axs[1].plot(history["train_loss"], label="train error")
    axs[1].plot(history["valid_loss"], label="valid error")
    axs[1].set_ylabel("Error")
    axs[1].set_xlabel("Epoch")
    axs[1].legend(loc="best")
    axs[1].set_title("Error eval")
 
    fig.tight_layout()
    plt.show()

## Neural Networks go brrrrrr...... 

### Training Setup

In [14]:
from torchinfo import summary 

# compile network 
model = UrbanSoundCNN().to(device)

# weight_decay = l2 regularization in pytorch 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)

# loss function
loss_fn = nn.CrossEntropyLoss()

# this moves the model to the cpu 
summary(model, input_size=(1, 1, 64, 44))

Layer (type:depth-idx)                   Output Shape              Param #
UrbanSoundCNN                            [1, 10]                   --
├─Sequential: 1-1                        [1, 10]                   --
│    └─Conv2d: 2-1                       [1, 16, 64, 44]           160
│    └─BatchNorm2d: 2-2                  [1, 16, 64, 44]           32
│    └─ReLU: 2-3                         [1, 16, 64, 44]           --
│    └─MaxPool2d: 2-4                    [1, 16, 32, 22]           --
│    └─Conv2d: 2-5                       [1, 32, 32, 22]           4,640
│    └─BatchNorm2d: 2-6                  [1, 32, 32, 22]           64
│    └─ReLU: 2-7                         [1, 32, 32, 22]           --
│    └─MaxPool2d: 2-8                    [1, 32, 16, 11]           --
│    └─Conv2d: 2-9                       [1, 64, 17, 12]           8,256
│    └─BatchNorm2d: 2-10                 [1, 64, 17, 12]           128
│    └─ReLU: 2-11                        [1, 64, 17, 12]           --
│    └─

In [15]:
train_dataset = UrbanSoundDataset(train_annotations, audio_dir, make_mel(), SAMPLE_RATE, NUM_SAMPLES, device)
valid_dataset = UrbanSoundDataset(valid_annotations, audio_dir, make_mel(), SAMPLE_RATE, NUM_SAMPLES, device)
test_dataset  = UrbanSoundDataset(test_annotations,  audio_dir, make_mel(), SAMPLE_RATE, NUM_SAMPLES, device)

train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=128)
test_dataloader  = DataLoader(test_dataset,  batch_size=128)

# print(f"Model device: {next(model.parameters()).device}")
# print(f"Device variable: {device}")
# signal, label = train_dataset[0]
# print(f"Sample device: {signal.device}")

# train network 
model = model.to(device)
model_training_history = train(model, train_dataloader, valid_dataloader, optimizer, loss_fn, device, epochs=10)

/Users/pranavrajan/Desktop/ml-engineering-practice/.venv/lib/python3.10/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [1, 44, 513]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


Epoch 1/10 | train_loss: 2.1783 - train_acc: 0.2113 | valid_loss: 2.1436 - valid_acc: 0.2402


KeyboardInterrupt: 

### Overfitting Debugging

In [ ]:
# plot accuracy and error over the epochs 
plot_history(model_training_history) 

### Evaluation 

In [ ]:
test_loss, test_acc = evaluate(model, test_dataloader, loss_fn, device)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

test_loss, test_acc = evaluate(model, test_dataloader, loss_fn, device)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

## Inference

### GTZAN Inference

In [ ]:
# def predict(model, X, y):
#     X = X.unsqueeze(0).to(device)
#     model.eval() 
#     with torch.no_grad():
#         logits = model(X)
#         predicted_class = logits.argmax(1).item()

#     target_genre = mapping[y.item()]
#     predicted_genre = mapping[predicted_class]
#     print(f"Prediction: {predicted_class} - {predicted_genre} | Target: {y.item()} - {target_genre}")

In [ ]:
# X, y = test_dataset[100]
# predict(model, X, y)

### MNIST Inference

In [ ]:
# def predict(model, X, y):
#     class_mapping = [str(i) for i in range(10)]
#     X = X.unsqueeze(0).to(device)
#     model.eval()
#     with torch.no_grad():
#         logits = model(X)   # add batch dim
#         predicted = class_mapping[logits.argmax(1).item()]
#         expected  = class_mapping[y]
#     print(f"Predicted: '{predicted}' | Expected: '{expected}'")

In [ ]:
# X, y = test_data[0]
# predict(model, X, y)

### UrbanSound Inference